In [0]:
use catalog identifier(:catalog)

In [0]:
CREATE OR REPLACE TABLE ops.time_travel_check AS
SELECT
  cur.charger_id,
  prev.max_kw     AS max_kw_before_batch2,
  cur.max_kw      AS max_kw_current,
  prev.is_active  AS is_active_before_batch2,
  cur.is_active   AS is_active_current
FROM silver.chargers AS cur
LEFT JOIN (
  SELECT charger_id, max_kw, is_active
  FROM table_changes('silver.chargers', 2)
  WHERE _change_type = 'update_preimage'
    AND _commit_version = (SELECT max(_commit_version) FROM table_changes('silver.chargers', 2))
) AS prev
  ON prev.charger_id = cur.charger_id
WHERE cur.charger_id IN ('CH004', 'CH011', 'CH027');


CREATE OR REPLACE TABLE ops.charger_changes_batch2 AS
SELECT
  charger_id,
  _change_type    AS change_type,
  max_kw,
  is_active,
  _commit_version AS commit_version
FROM table_changes('silver.chargers', 2)
WHERE _commit_version = (
  SELECT max(_commit_version)
  FROM table_changes('silver.chargers', 2)
);


ALTER TABLE gold.fact_sessions CLUSTER BY (date_key, station_id);
OPTIMIZE gold.fact_sessions;


CREATE OR REPLACE TABLE ops.sessions_drill DEEP CLONE silver.sessions;

DELETE FROM ops.sessions_drill
WHERE charger_id IN (SELECT charger_id FROM silver.chargers WHERE station_id = 'ST01');

RESTORE TABLE ops.sessions_drill TO VERSION AS OF 0;


In [0]:
select * from vp_ds20010904.ops.time_travel_check